In [26]:
# [CELL 1] IMPORT REQUIRED LIBRARIES

import csv
import json
import shutil
from pathlib import Path
from collections import Counter

In [27]:
# [CELL 2] STUDENT-SPECIFIC CONFIGURATION

LAST_NAME = "Correa"
STUDENT_ID = "TUPM-26-1106"
SEED_DIGIT = 6
ID_SUM = 16

TARGET_YEAR = 2015 + SEED_DIGIT

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Target Year:", TARGET_YEAR)

Student: Correa
Student ID: TUPM-26-1106
Seed Digit: 6
ID Sum: 16
Target Year: 2021


In [28]:
# [CELL 3] CREATE DIRECTORIES AND DEFINE FILE PATHS

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

INPUT_FILE = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

FILTERED_FILE = OUTPUT_DIR / "filtered_cybersecurity_incidents.csv"
TOP5_FILE = OUTPUT_DIR / "top_costliest_incidents.json"
SUMMARY_FILE = OUTPUT_DIR / "incident_summary.json"
BACKUP_FILE = BACKUP_DIR / "top_costliest_incidents_backup.json"
REPORT_FILE = OUTPUT_DIR / "cybersecurity_incident_report.txt"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Project Directory:", BASE_DIR)
print("Input File:", INPUT_FILE)
print("Input File Exists:", INPUT_FILE.exists())
print("Output Directory:", OUTPUT_DIR)
print("Backup Directory:", BACKUP_DIR)

Project Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises
Input File: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/data/Global_Cybersecurity_Threats_2015-2024.csv
Input File Exists: True
Output Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/output
Backup Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/backup


In [29]:
# [CELL 4] INSPECT DATASET AND PREVIEW RECORDS

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    print("===== DATASET COLUMNS =====")
    print(reader.fieldnames)

    print("\n===== FIRST RECORD =====")
    first_record = next(reader, None)
    print(first_record)

===== DATASET COLUMNS =====
['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']

===== FIRST RECORD =====
{'Country': 'China', 'Year': '2019', 'Attack Type': 'Phishing', 'Target Industry': 'Education', 'Financial Loss (in Million $)': '80.53', 'Number of Affected Users': '773169', 'Attack Source': 'Hacker Group', 'Security Vulnerability Type': 'Unpatched Software', 'Defense Mechanism Used': 'VPN', 'Incident Resolution Time (in Hours)': '63'}


In [30]:
# [CELL 5] READ AND FILTER CYBERSECURITY INCIDENTS

selected_incidents = []

total_records = 0
invalid_year_records = 0
invalid_financial_records = 0

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        try:
            year = int(row["Year"])
        except (ValueError, TypeError):
            invalid_year_records += 1
            continue

        if year != TARGET_YEAR:
            continue

        try:
            financial_loss = float(row["Financial Loss (in Million $)"])

            if financial_loss < 0:
                raise ValueError

        except (ValueError, TypeError):
            invalid_financial_records += 1
            continue

        incident = {
            "Country": row["Country"],
            "Year": year,
            "Attack Type": row["Attack Type"],
            "Target Industry": row["Target Industry"],
            "Financial Loss (in Million $)": financial_loss,
            "Number of Affected Users": row["Number of Affected Users"],
            "Attack Source": row["Attack Source"],
            "Security Vulnerability Type": row["Security Vulnerability Type"],
            "Defense Mechanism Used": row["Defense Mechanism Used"],
            "Incident Resolution Time (in Hours)": row["Incident Resolution Time (in Hours)"]
        }

        selected_incidents.append(incident)

print("Total Records:", total_records)
print("Target Year:", TARGET_YEAR)
print("Incidents in Target Year:", len(selected_incidents))
print("Invalid Year Records:", invalid_year_records)
print("Invalid Financial Records:", invalid_financial_records)

Total Records: 3000
Target Year: 2021
Incidents in Target Year: 299
Invalid Year Records: 0
Invalid Financial Records: 0


In [31]:
# [CELL 6] COUNT INCIDENTS PER ATTACK TYPE

attack_type_counts = Counter(
    incident["Attack Type"] for incident in selected_incidents
)

print("===== INCIDENT COUNT PER ATTACK TYPE =====")

for attack_type, count in sorted(attack_type_counts.items()):
    print(f"{attack_type}: {count}")

===== INCIDENT COUNT PER ATTACK TYPE =====
DDoS: 47
Malware: 49
Man-in-the-Middle: 48
Phishing: 41
Ransomware: 55
SQL Injection: 59


In [32]:
# [CELL 7] CALCULATE TOTAL AND AVERAGE FINANCIAL LOSS

if selected_incidents:
    total_financial_loss = sum(
        incident["Financial Loss (in Million $)"]
        for incident in selected_incidents
    )

    average_financial_loss = (
        total_financial_loss / len(selected_incidents)
    )

else:
    total_financial_loss = 0
    average_financial_loss = None

print("Total Financial Loss:", round(total_financial_loss, 2), "USD million")

if average_financial_loss is not None:
    print("Average Financial Loss:", round(average_financial_loss, 2), "USD million")
else:
    print("Average Financial Loss: No matching incidents")

Total Financial Loss: 15873.41 USD million
Average Financial Loss: 53.09 USD million


In [33]:
# [CELL 8] IDENTIFY FIVE COSTLIEST INCIDENTS

# Sorting is stable, so incidents with tied losses
# retain their original dataset order.

top_five_incidents = sorted(
    selected_incidents,
    key=lambda incident: incident["Financial Loss (in Million $)"],
    reverse=True
)[:5]

print("===== FIVE COSTLIEST INCIDENTS =====")

for rank, incident in enumerate(top_five_incidents, start=1):
    print(f"\nRank {rank}")
    print("Country:", incident["Country"])
    print("Attack Type:", incident["Attack Type"])
    print("Target Industry:", incident["Target Industry"])
    print("Financial Loss:", incident["Financial Loss (in Million $)"], "USD million")

===== FIVE COSTLIEST INCIDENTS =====

Rank 1
Country: USA
Attack Type: DDoS
Target Industry: Telecommunications
Financial Loss: 99.81 USD million

Rank 2
Country: France
Attack Type: Ransomware
Target Industry: IT
Financial Loss: 99.77 USD million

Rank 3
Country: France
Attack Type: SQL Injection
Target Industry: Government
Financial Loss: 98.76 USD million

Rank 4
Country: China
Attack Type: SQL Injection
Target Industry: IT
Financial Loss: 98.57 USD million

Rank 5
Country: India
Attack Type: SQL Injection
Target Industry: IT
Financial Loss: 98.09 USD million


In [34]:
# [CELL 9] EXPORT FIVE COSTLIEST INCIDENTS TO JSON

with open(TOP5_FILE, "w", encoding="utf-8") as file:
    json.dump(top_five_incidents, file, indent=4)

print("Top Five JSON File Created:", TOP5_FILE.exists())
print("Number of Incidents Exported:", len(top_five_incidents))

Top Five JSON File Created: True
Number of Incidents Exported: 5


In [35]:
# [CELL 10] EXPORT SELECTED INCIDENTS TO FILTERED CSV

fieldnames = [
    "Country",
    "Year",
    "Attack Type",
    "Target Industry",
    "Financial Loss (in Million $)",
    "Number of Affected Users",
    "Attack Source",
    "Security Vulnerability Type",
    "Defense Mechanism Used",
    "Incident Resolution Time (in Hours)"
]

with open(FILTERED_FILE, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(selected_incidents)

print("Filtered CSV Created:", FILTERED_FILE.exists())
print("Selected Incidents Exported:", len(selected_incidents))

Filtered CSV Created: True
Selected Incidents Exported: 299


In [36]:
# [CELL 11] CREATE JSON SUMMARY REPORT

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "incidents_in_target_year": len(selected_incidents),
    "invalid_year_records": invalid_year_records,
    "invalid_financial_records": invalid_financial_records,
    "count_per_attack_type": dict(sorted(attack_type_counts.items())),
    "total_financial_loss_USD_millions": round(total_financial_loss, 2),
    "average_financial_loss_USD_millions": (
        round(average_financial_loss, 2)
        if average_financial_loss is not None else None
    ),
    "five_costliest_incidents": top_five_incidents
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON Summary Created:", SUMMARY_FILE.exists())

JSON Summary Created: True


In [37]:
# [CELL 12] CREATE AND VERIFY JSON BACKUP

shutil.copy2(TOP5_FILE, BACKUP_FILE)

backup_exists = BACKUP_FILE.exists()

backup_size = BACKUP_FILE.stat().st_size if backup_exists else 0

backup_verified = (
    backup_exists
    and TOP5_FILE.read_bytes() == BACKUP_FILE.read_bytes()
)

print("Backup Exists:", backup_exists)
print("Backup Verified:", backup_verified)
print("Backup File Size:", backup_size, "bytes")

Backup Exists: True
Backup Verified: True
Backup File Size: 2142 bytes


In [38]:
# [CELL 13] GENERATE CYBERSECURITY INCIDENT TEXT REPORT

report_lines = []

report_lines.append("CYBERSECURITY INCIDENT REPORT")
report_lines.append("=" * 40)
report_lines.append(f"Student: {LAST_NAME}")
report_lines.append(f"Student ID: {STUDENT_ID}")
report_lines.append(f"Target Year: {TARGET_YEAR}")

report_lines.append("\nDATASET SUMMARY")
report_lines.append("-" * 40)
report_lines.append(f"Total Records: {total_records}")
report_lines.append(f"Incidents in Target Year: {len(selected_incidents)}")
report_lines.append(f"Invalid Year Records: {invalid_year_records}")
report_lines.append(f"Invalid Financial Records: {invalid_financial_records}")

report_lines.append("\nINCIDENT COUNT PER ATTACK TYPE")
report_lines.append("-" * 40)

if attack_type_counts:
    for attack_type, count in sorted(attack_type_counts.items()):
        report_lines.append(f"{attack_type}: {count}")
else:
    report_lines.append("No matching incidents.")

report_lines.append("\nFINANCIAL LOSS SUMMARY")
report_lines.append("-" * 40)
report_lines.append(
    f"Total Financial Loss: {total_financial_loss:.2f} USD million"
)

if average_financial_loss is not None:
    report_lines.append(
        f"Average Financial Loss: {average_financial_loss:.2f} USD million"
    )
else:
    report_lines.append("Average Financial Loss: No matching incidents.")

report_lines.append("\nFIVE COSTLIEST INCIDENTS")
report_lines.append("-" * 40)

if top_five_incidents:
    for rank, incident in enumerate(top_five_incidents, start=1):
        report_lines.append(
            f"{rank}. {incident['Country']} | "
            f"{incident['Attack Type']} | "
            f"{incident['Target Industry']} | "
            f"{incident['Financial Loss (in Million $)']:.2f} USD million"
        )
else:
    report_lines.append("No matching incidents.")

report_lines.append("\nOUTPUT FILE INFORMATION")
report_lines.append("-" * 40)

output_files = [
    FILTERED_FILE,
    TOP5_FILE,
    SUMMARY_FILE,
    BACKUP_FILE,
    REPORT_FILE
]

for output_file in output_files:
    if output_file.exists():
        report_lines.append(
            f"{output_file.name}: {output_file.stat().st_size} bytes"
        )
    else:
        report_lines.append(f"{output_file.name}: Not yet created")

report_lines.append("\nBACKUP VERIFICATION")
report_lines.append("-" * 40)
report_lines.append(f"Backup Exists: {backup_exists}")
report_lines.append(f"Backup Verified: {backup_verified}")

with open(REPORT_FILE, "w", encoding="utf-8") as file:
    file.write("\n".join(report_lines))

print("Text Report Created:", REPORT_FILE.exists())
print("\n===== CYBERSECURITY INCIDENT REPORT =====")
print(REPORT_FILE.read_text(encoding="utf-8"))

Text Report Created: True

===== CYBERSECURITY INCIDENT REPORT =====
CYBERSECURITY INCIDENT REPORT
Student: Correa
Student ID: TUPM-26-1106
Target Year: 2021

DATASET SUMMARY
----------------------------------------
Total Records: 3000
Incidents in Target Year: 299
Invalid Year Records: 0
Invalid Financial Records: 0

INCIDENT COUNT PER ATTACK TYPE
----------------------------------------
DDoS: 47
Malware: 49
Man-in-the-Middle: 48
Phishing: 41
Ransomware: 55
SQL Injection: 59

FINANCIAL LOSS SUMMARY
----------------------------------------
Total Financial Loss: 15873.41 USD million
Average Financial Loss: 53.09 USD million

FIVE COSTLIEST INCIDENTS
----------------------------------------
1. USA | DDoS | Telecommunications | 99.81 USD million
2. France | Ransomware | IT | 99.77 USD million
3. France | SQL Injection | Government | 98.76 USD million
4. China | SQL Injection | IT | 98.57 USD million
5. India | SQL Injection | IT | 98.09 USD million

OUTPUT FILE INFORMATION
---------------

In [39]:
# [CELL 14] VERIFY ALL OUTPUT FILES

# Verify filtered CSV row count
with open(FILTERED_FILE, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    exported_count = sum(1 for row in reader)

count_verified = exported_count == len(selected_incidents)

# Verify top five JSON record count
with open(TOP5_FILE, "r", encoding="utf-8") as file:
    exported_top_five = json.load(file)

top_five_verified = len(exported_top_five) == len(top_five_incidents)

# Verify JSON backup
backup_verified = (
    BACKUP_FILE.exists()
    and TOP5_FILE.read_bytes() == BACKUP_FILE.read_bytes()
)

print("===== FINAL FILE VERIFICATION =====")
print("Filtered CSV Exists:", FILTERED_FILE.exists())
print("CSV Expected Records:", len(selected_incidents))
print("CSV Exported Records:", exported_count)
print("CSV Count Verified:", count_verified)

print("\nTop Five JSON Exists:", TOP5_FILE.exists())
print("Top Five JSON Verified:", top_five_verified)

print("\nSummary JSON Exists:", SUMMARY_FILE.exists())
print("Backup Exists:", BACKUP_FILE.exists())
print("Backup Verified:", backup_verified)
print("Backup File Size:", BACKUP_FILE.stat().st_size, "bytes" if BACKUP_FILE.exists() else "N/A")
print("Text Report Exists:", REPORT_FILE.exists())

===== FINAL FILE VERIFICATION =====
Filtered CSV Exists: True
CSV Expected Records: 299
CSV Exported Records: 299
CSV Count Verified: True

Top Five JSON Exists: True
Top Five JSON Verified: True

Summary JSON Exists: True
Backup Exists: True
Backup Verified: True
Backup File Size: 2142 bytes
Text Report Exists: True
